# 09. Gene Subsampling Bootstrap

This notebook is standalone and reconstructs everything needed from previous outputs.

Goal: test robustness of patient-level SICI signal when each program gene set is randomly subsampled.


In [10]:
import os
from pathlib import Path

# Resolve project root robustly (works from notebooks/, project root, or Downloads/).
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Spatial HCC",
    Path.cwd().parent / "Spatial HCC",
]
for _root in _candidates:
    if (_root / "st_adata_scored.h5ad").exists() or (_root / "GSE238264_RAW").exists():
        os.chdir(_root)
        break

print("Working directory:", Path.cwd())


Working directory: /Users/prateek/Downloads/Spatial HCC


In [11]:
import random
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import scipy.sparse as sp
from scipy.stats import spearmanr

RUN_MODE = os.environ.get("HCC_RUN_MODE", "full").strip().lower()
N_ITER_DEFAULT = int(os.environ.get("N_ITER_BOOT", "300" if RUN_MODE == "full" else "8"))
KEEP_FRAC_DEFAULT = float(os.environ.get("KEEP_FRAC_BOOT", "0.7"))
SEED_DEFAULT = int(os.environ.get("SEED_BOOT", "13"))

print("RUN_MODE:", RUN_MODE)
print("Default n_iter:", N_ITER_DEFAULT)
print("Default keep_frac:", KEEP_FRAC_DEFAULT)


RUN_MODE: full
Default n_iter: 300
Default keep_frac: 0.7


In [12]:
# Load scored spatial object.
if "st_adata" not in globals():
    h5_path = Path("st_adata_scored.h5ad")
    if not h5_path.exists():
        raise FileNotFoundError("st_adata_scored.h5ad not found. Run notebook 02 first.")
    st_adata = sc.read_h5ad(h5_path.as_posix())

required_obs = ["sample", "spot_exhaustion", "spot_cytotoxic", "spot_tam", "spot_caf", "spot_malignant", "spot_bcell"]
missing_obs = [c for c in required_obs if c not in st_adata.obs.columns]
if missing_obs:
    raise KeyError(f"st_adata.obs missing required columns: {missing_obs}")

st_adata.obs["sample"] = st_adata.obs["sample"].astype(str)
print("st_adata:", st_adata)
print("Samples:", sorted(st_adata.obs["sample"].unique()))


st_adata: AnnData object with n_obs × n_vars = 17292 × 22689
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'spot_exhaustion', 'spot_cytotoxic', 'spot_tam', 'response', 'spot_caf', 'spot_malignant', 'spot_stemness', 'spot_bcell', 'spot_plasma', 'spot_endo'
    var: 'n_cells'
    uns: 'log1p'
    obsm: 'spatial'
    layers: 'counts'
Samples: ['HCC1R', 'HCC2R', 'HCC3R', 'HCC4R', 'HCC5NR', 'HCC6NR', 'HCC7NR']


In [13]:
def normalize_group_label(x):
    """Normalize labels to R/NR."""
    s = str(x).strip().upper()
    if s in {"R", "RESPONDER"}:
        return "R"
    if s in {"NR", "NONRESPONDER", "NON-RESPONDER", "NON_RESPONDER"}:
        return "NR"
    return s


# Build reference df_patient from previous outputs.
required_cols = ["sample", "group", "C_b_cyto", "C_malig_caf", "C_exh_cyto"]
df_patient_ref = None
source_name = None

candidate_paths = [
    Path("sensitivity_outputs/df_patient_for_sici_sensitivity.csv"),
    Path("SICI_sensitivity_outputs/df_patient_for_sici_sensitivity.csv"),
    Path("supplement_exports/Table_S1_metrics_couplings_torus.csv"),
    Path("supplement_exports_split/Table_S1_master_metrics_baselines.csv"),
    Path("supplement_exports_v2/Table_S1_master_metrics_baselines_torus.csv"),
]

for cp in candidate_paths:
    if cp.exists():
        tmp = pd.read_csv(cp)
        if all(c in tmp.columns for c in required_cols):
            df_patient_ref = tmp.copy()
            source_name = str(cp)
            break

if df_patient_ref is None:
    raise FileNotFoundError(
        "Could not load df_patient reference from prior outputs. "
        "Run notebook 03 and/or notebook 06 first."
    )

df_patient_ref = df_patient_ref[required_cols + (["SICI"] if "SICI" in df_patient_ref.columns else [])].copy()
df_patient_ref["sample"] = df_patient_ref["sample"].astype(str)
df_patient_ref["group"] = df_patient_ref["group"].map(normalize_group_label)
df_patient_ref = df_patient_ref[df_patient_ref["group"].isin(["R", "NR"])].copy()

if df_patient_ref["sample"].duplicated().any():
    agg = {
        "group": "first",
        "C_b_cyto": "mean",
        "C_malig_caf": "mean",
        "C_exh_cyto": "mean",
    }
    if "SICI" in df_patient_ref.columns:
        agg["SICI"] = "mean"
    df_patient_ref = df_patient_ref.groupby("sample", as_index=False).agg(agg)

if "SICI" not in df_patient_ref.columns:
    df_patient_ref["SICI"] = (
        df_patient_ref["C_b_cyto"]
        - df_patient_ref["C_malig_caf"]
        - df_patient_ref["C_exh_cyto"]
    )

# Sample -> group mapping for recomputed tables.
sample_groups = dict(zip(df_patient_ref["sample"], df_patient_ref["group"]))

print("df_patient reference source:", source_name)
display(df_patient_ref.sort_values("sample"))

# Compatibility alias expected by downstream bootstrap function.
df_patient = df_patient_ref.copy()


df_patient reference source: sensitivity_outputs/df_patient_for_sici_sensitivity.csv


,sample,group,C_b_cyto,C_malig_caf,C_exh_cyto,SICI
0,HCC1R,R,0.017711,-0.033758,0.019049,0.032420
1,HCC2R,R,-0.032210,-0.057423,0.038262,-0.013049
2,HCC3R,R,0.007831,-0.040782,0.022126,0.026488
3,HCC4R,R,0.095418,-0.035658,0.061939,0.069138
4,HCC5NR,NR,-0.010910,-0.065038,-0.025713,0.079841
5,HCC6NR,NR,0.002226,-0.104665,0.032672,0.074218
6,HCC7NR,NR,0.032413,-0.070682,0.010254,0.092841


In [14]:
# Gene pools from previous notebooks.
exhaustion_genes = ["PDCD1", "LAG3", "HAVCR2", "TIGIT", "CTLA4", "TOX"]
cytotoxic_genes = ["GZMB", "GZMK", "PRF1", "NKG7", "GNLY"]
tam_genes = ["C1QC", "C1QB", "C1QA", "APOE", "LGALS3", "SPP1", "MRC1", "CD163"]
caf_genes = ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "TAGLN", "ACTA2"]
malignant_genes = ["EPCAM", "KRT19", "KRT8", "KRT18", "MSLN", "GPC3", "ALB"]
stemness_genes = ["EPCAM", "PROM1", "KRT19", "SOX9", "ALDH1A1", "CD24"]
bcell_genes = ["MS4A1", "CD79A", "CD74", "HLA-DRA", "BANK1", "CD37"]
plasma_genes = ["MZB1", "XBP1", "JCHAIN", "IGHG1", "IGHG3", "IGKC"]
endo_genes = ["PECAM1", "VWF", "KDR", "RAMP2", "EMCN"]

gene_dict = {
    "exhaustion": exhaustion_genes,
    "cytotoxic": cytotoxic_genes,
    "tam": tam_genes,
    "caf": caf_genes,
    "malignant": malignant_genes,
    "stemness": stemness_genes,
    "bcell": bcell_genes,
    "plasma": plasma_genes,
    "endo": endo_genes,
}

for k, genes in gene_dict.items():
    present = [g for g in genes if g in st_adata.var_names]
    print(f"{k:10s}: {len(present):2d}/{len(genes):2d} genes present")


exhaustion:  6/ 6 genes present
cytotoxic :  5/ 5 genes present
tam       :  8/ 8 genes present
caf       :  7/ 7 genes present
malignant :  7/ 7 genes present
stemness  :  6/ 6 genes present
bcell     :  6/ 6 genes present
plasma    :  6/ 6 genes present
endo      :  5/ 5 genes present


In [15]:
def _build_sample_cache(ad_all, sample_groups_map):
    """Precompute per-sample graph edges and expression handles once."""
    from scipy.sparse import triu

    cache = {}
    sample_series = ad_all.obs["sample"].astype(str)
    unique_samples = sorted(sample_series.unique())

    for sample in unique_samples:
        ad = ad_all[sample_series == sample].copy()
        sq.gr.spatial_neighbors(ad, coord_type="grid")
        A = ad.obsp["spatial_connectivities"]

        U = triu(A, k=1).tocoo()
        i = U.row.astype(np.int64)
        j = U.col.astype(np.int64)

        cache[sample] = {
            "X": ad.X,
            "edges_i": i,
            "edges_j": j,
            "group": sample_groups_map.get(sample),
            "n_spots": ad.n_obs,
        }

    return cache


def _score_program(X, gene_indices):
    """Program score = mean expression over selected genes, z-scored within sample."""
    if len(gene_indices) < 2:
        return np.full(X.shape[0], np.nan, dtype=float)

    sub = X[:, gene_indices]
    if sp.issparse(sub):
        score = np.asarray(sub.mean(axis=1)).ravel().astype(float)
    else:
        score = np.asarray(sub, dtype=float).mean(axis=1)

    mu = np.nanmean(score)
    sd = np.nanstd(score)
    return (score - mu) / (sd + 1e-9)


def _coupling_from_fields(f1, f2, edges_i, edges_j):
    """Pearson coupling between edge gradients of two fields."""
    da = f1[edges_i] - f1[edges_j]
    db = f2[edges_i] - f2[edges_j]

    valid = np.isfinite(da) & np.isfinite(db)
    if valid.sum() < 10:
        return np.nan

    da = da[valid]
    db = db[valid]
    if np.nanstd(da) < 1e-12 or np.nanstd(db) < 1e-12:
        return np.nan

    return float(np.corrcoef(da, db)[0, 1])


def _compute_patient_metrics_fast(sample_cache, program_gene_indices):
    """Compute patient-level couplings/SICI using precomputed cache + gene index sets."""
    rows = []
    for sample, item in sample_cache.items():
        X = item["X"]
        i = item["edges_i"]
        j = item["edges_j"]

        f_exh = _score_program(X, program_gene_indices["exhaustion"])
        f_cyto = _score_program(X, program_gene_indices["cytotoxic"])
        f_tam = _score_program(X, program_gene_indices["tam"])
        f_caf = _score_program(X, program_gene_indices["caf"])
        f_malig = _score_program(X, program_gene_indices["malignant"])
        f_b = _score_program(X, program_gene_indices["bcell"])

        c_b_cyto = _coupling_from_fields(f_b, f_cyto, i, j)
        c_malig_caf = _coupling_from_fields(f_malig, f_caf, i, j)
        c_exh_cyto = _coupling_from_fields(f_exh, f_cyto, i, j)

        rows.append({
            "sample": sample,
            "group": item["group"],
            "C_b_cyto": c_b_cyto,
            "C_malig_caf": c_malig_caf,
            "C_exh_cyto": c_exh_cyto,
            "SICI": c_b_cyto - c_malig_caf - c_exh_cyto,
        })

    out = pd.DataFrame(rows)
    out = out[out["group"].isin(["R", "NR"])].reset_index(drop=True)
    return out


# Precompute cache and gene index mapping once for fast recomputation.
_SAMPLE_CACHE = _build_sample_cache(st_adata, sample_groups)
_VAR_NAMES = np.asarray(st_adata.var_names, dtype=str)
_GENE_TO_IDX = {g: i for i, g in enumerate(_VAR_NAMES)}


def recompute_metrics(boot_genes):
    """Recompute patient-level metrics from a gene dictionary (gene names)."""
    program_gene_indices = {}
    for prog, genes in boot_genes.items():
        program_gene_indices[prog] = [_GENE_TO_IDX[g] for g in genes if g in _GENE_TO_IDX]

    return _compute_patient_metrics_fast(_SAMPLE_CACHE, program_gene_indices)


def bootstrap_single_program(program_key, gene_dict, n_iter=200):
    results = []

    original_sici = df_patient["SICI"].values

    for i in range(n_iter):
        boot_genes = {k: list(v) for k, v in gene_dict.items()}

        genes = [g for g in gene_dict[program_key] if g in _GENE_TO_IDX]
        n_keep = max(2, int(len(genes) * 0.7))
        n_keep = min(n_keep, len(genes))

        if n_keep < 2:
            results.append(np.nan)
            continue

        boot_genes[program_key] = random.sample(genes, n_keep)

        df_boot = recompute_metrics(boot_genes)

        mean_R = df_boot[df_boot["group"] == "R"]["SICI"].mean()
        mean_NR = df_boot[df_boot["group"] == "NR"]["SICI"].mean()

        diff = mean_R - mean_NR

        results.append(diff)

    return pd.Series(results)


INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           


In [ ]:
# Run single-program bootstrap for requested programs.
program_runs = [
    ("exhaustion", "exhaustion"),
    ("cytotoxic", "cytotoxic"),
    ("caf", "CAF"),
]

boot_tables = []
summary_rows = []

for program_key, program_label in program_runs:
    s = bootstrap_single_program(program_key, gene_dict, n_iter=N_ITER_DEFAULT)
    d = s.to_frame(name="diff_R_minus_NR")
    d["program_key"] = program_key
    d["program_label"] = program_label
    d["iter"] = np.arange(len(d), dtype=int)
    boot_tables.append(d)

    summary_rows.append({
        "program_key": program_key,
        "program_label": program_label,
        "n_iter": int(len(d)),
        "keep_frac": 0.7,
        "fraction_positive_diff": float((d["diff_R_minus_NR"] > 0).mean()) if len(d) else np.nan,
        "median_diff": float(d["diff_R_minus_NR"].median()) if len(d) else np.nan,
    })

df_boot_all = pd.concat(boot_tables, ignore_index=True)
summary = pd.DataFrame(summary_rows).sort_values("program_label").reset_index(drop=True)

print("Total bootstrap rows:", len(df_boot_all))
display(summary)
display(df_boot_all.head())


In [ ]:
# Save outputs.
out_dir = Path("gene_subsampling_bootstrap_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

df_boot_all.to_csv(out_dir / "gene_subsampling_bootstrap_results_all_programs.csv", index=False)
summary.to_csv(out_dir / "gene_subsampling_bootstrap_summary_all_programs.csv", index=False)

# Keep backward-compatible file names too.
df_boot_all.to_csv(out_dir / "gene_subsampling_bootstrap_results.csv", index=False)
summary.to_csv(out_dir / "gene_subsampling_bootstrap_summary.csv", index=False)

print("Saved:")
print("-", out_dir / "gene_subsampling_bootstrap_results_all_programs.csv")
print("-", out_dir / "gene_subsampling_bootstrap_summary_all_programs.csv")
print("-", out_dir / "gene_subsampling_bootstrap_results.csv")
print("-", out_dir / "gene_subsampling_bootstrap_summary.csv")
